# LangChain：带工具调用的多模态（图片）Agent

本笔记演示如何把 **图片** 作为用户消息交给 **支持 Vision 的聊天模型**，并保留 **工具调用**（function calling）能力。

## 运行前准备

1. **模型必须支持图像输入**（例如 LM Studio 加载 VL 模型、或 OpenAI `gpt-4o` 等）。纯文本模型会报错或忽略图。
2. 依赖与 `06_langchain_agent_react.ipynb` 相同目录下的虚拟环境即可（`langchain`、`langchain-openai`、`langchain-core` 等）。
3. 下面默认沿用本地 **LM Studio** OpenAI 兼容接口；若走官方 OpenAI，把 `base_url` 改为默认并设置 `OPENAI_API_KEY`，`model` 改为如 `gpt-4o`。


In [ ]:
import base64
import mimetypes
from pathlib import Path
from typing import Union

from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

LM_STUDIO_BASE = "http://127.0.0.1:1234/v1"
llm = ChatOpenAI(
    base_url=LM_STUDIO_BASE,
    api_key="lm-studio",
    temperature=0,
    model="local",
)


def image_path_to_data_url(path: Union[str, Path]) -> str:
    """本地图片 → data URL，便于 OpenAI 兼容接口传图。"""
    p = Path(path).expanduser().resolve()
    raw = p.read_bytes()
    mime = mimetypes.guess_type(p.name)[0] or "image/png"
    b64 = base64.standard_b64encode(raw).decode("ascii")
    return f"data:{mime};base64,{b64}"


@tool
def multiply(a: float, b: float) -> float:
    """计算 a * b。当用户需要根据图中数字做乘法时使用。"""
    return float(a) * float(b)


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是能阅读图片的助手，并可在需要时调用工具。用中文回答。",
        ),
        MessagesPlaceholder("chat_history", optional=True),
        MessagesPlaceholder("user_message"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

tools = [multiply]
agent_runnable = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent_runnable,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=8,
)

## 调用示例

- `IMAGE_PATH`：指向本机图片；设为 `None` 时使用下方 **1×1 占位 PNG**（仅用于打通请求，谈不上真实视觉理解）。
- 也可把 `data_url` 换成公网 `https://...` 图片地址（`image_url` 块里直接写 URL）。


In [ ]:
IMAGE_PATH = None  # 例如："/Users/you/Pictures/demo.png"

DEMO_TINY_PNG = (
    "data:image/png;base64,"
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg=="
)

if IMAGE_PATH:
    data_url = image_path_to_data_url(IMAGE_PATH)
else:
    data_url = DEMO_TINY_PNG

user = HumanMessage(
    content=[
        {
            "type": "text",
            "text": (
                "简要描述这张图片。如果图里有两个清晰的阿拉伯数字，"
                "请先读出它们，再用 multiply 工具计算乘积并在回答里写出结果。"
            ),
        },
        {"type": "image_url", "image_url": {"url": data_url}},
    ]
)

result = agent_executor.invoke(
    {
        "chat_history": [],
        "user_message": [user],
    }
)
print(result["output"])